In [5]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler, LabelEncoder

import tensorflow as tf
from tensorflow.keras import layers, models

ImportError: cannot import name 'runtime_version' from 'google.protobuf' (c:\Users\shrib\anaconda3\envs\esp_env\Lib\site-packages\google\protobuf\__init__.py)

In [ ]:
WINDOW_SIZE = 154
# Your specific column selection
SENSOR_COLS = ['flex_1', 'flex_2', 'flex_3', 'flex_4', 'flex_5', 'ACCx', 'ACCy', 'ACCz']
LABEL_COL = 'label' # Adjust if your CSV uses a different name for the gesture
SUBJECT_COL = 'subject'

In [ ]:
def prepare_data(csv_path):
    df = pd.read_csv(csv_path)
    
    # Encode labels to integers
    le = LabelEncoder()
    df[LABEL_COL] = le.fit_transform(df[LABEL_COL])
    
    # 1. Normalize sensor values (Critical for CNN convergence)
    scaler = StandardScaler()
    df[SENSOR_COLS] = scaler.fit_transform(df[SENSOR_COLS])
    
    X, y = [], []
    
    # 2. Windowing logic grouped by subject
    for _, group in df.groupby(SUBJECT_COL):
        sensors = group[SENSOR_COLS].values
        labels = group[LABEL_COL].values
        
        # Slide through the data
        for i in range(0, len(group) - WINDOW_SIZE, 77): # 50% overlap for more data
            window = sensors[i : i + WINDOW_SIZE]
            # Use the most frequent label in the window as the target
            target = np.bincount(labels[i : i + WINDOW_SIZE]).argmax()
            
            X.append(window)
            y.append(target)
    return np.array(X), np.array(y)

In [ ]:
X, y = prepare_data("dynamic_data.csv")
print(f"Dataset Shape: {X.shape}") # Goal: (Total_Windows, 154, 8)
print('Now doing next')

Dataset Shape: (13600, 154, 8)
Now doing next


In [ ]:
# Redefine the function with exact same code
def prepare_data(csv_path):
    df = pd.read_csv(csv_path)
    
    # Encode labels to integers
    le = LabelEncoder()
    df[LABEL_COL] = le.fit_transform(df[LABEL_COL])
    
    # 1. Normalize sensor values (Critical for CNN convergence)
    scaler = StandardScaler()
    df[SENSOR_COLS] = scaler.fit_transform(df[SENSOR_COLS])
    
    X, y = [], []
    
    # 2. Windowing logic grouped by subject
    for _, group in df.groupby(SUBJECT_COL):
        sensors = group[SENSOR_COLS].values
        labels = group[LABEL_COL].values
        
        # Slide through the data
        for i in range(0, len(group) - WINDOW_SIZE, 77): # 50% overlap for more data
            window = sensors[i : i + WINDOW_SIZE]
            # Use the most frequent label in the window as the target
            target = np.bincount(labels[i : i + WINDOW_SIZE]).argmax()
            
            X.append(window)
            y.append(target)
    return np.array(X), np.array(y)

# Test the redefined function
result = prepare_data("dynamic_data.csv")
print("Redefined function result type:", type(result))
if result is not None:
    X, y = result
    print(f"X shape: {X.shape}, y shape: {y.shape}")
else:
    print("Redefined function returned None")

Redefined function result type: <class 'tuple'>
X shape: (13600, 154, 8), y shape: (13600,)


In [ ]:
model = models.Sequential([
    layers.Input(shape=(154, 8)),
    layers.Conv1D(32, kernel_size=5, activation='relu'),
    layers.MaxPooling1D(pool_size=2),
    layers.Conv1D(64, kernel_size=3, activation='relu'),
    layers.GlobalAveragePooling1D(),
    layers.Dense(32, activation='relu'),
    layers.Dense(14, activation='softmax') # 14 Gesture Classes
])

In [ ]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# 2. Train (Assuming X, y from previous step)
model.fit(X, y, epochs=30, batch_size=32, validation_split=0.2)

Epoch 1/30
340/340 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.7950 - loss: 0.6677 - val_accuracy: 0.9235 - val_loss: 0.2633
Epoch 2/30
340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9607 - loss: 0.1307 - val_accuracy: 0.9272 - val_loss: 0.1996
Epoch 3/30
340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9730 - loss: 0.0853 - val_accuracy: 0.9357 - val_loss: 0.1925
Epoch 4/30
340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9795 - loss: 0.0668 - val_accuracy: 0.9408 - val_loss: 0.1886
Epoch 5/30
340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9822 - loss: 0.0557 - val_accuracy: 0.9632 - val_loss: 0.1063
Epoch 6/30
340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9839 - loss: 0.0489 - val_accuracy: 0.9566 - val_loss: 0.1145
Epoch 7/30
340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9869 - loss: 0.0391 - val_accuracy: 0.9500 - val_loss: 0.1563
Epoch 8/30
340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9902 - loss: 0.0329 - val_accuracy: 0.

In [ ]:
# Manually create ONNX model to match the TensorFlow architecture
import onnx
from onnx import helper, TensorProto, numpy_helper
import numpy as np

# Get the weights from the trained model
conv1_weights = model.layers[0].get_weights()[0]  # Conv1D weights
conv1_bias = model.layers[0].get_weights()[1]     # Conv1D bias

conv2_weights = model.layers[2].get_weights()[0]  # Conv1D weights
conv2_bias = model.layers[2].get_weights()[1]     # Conv1D bias

dense1_weights = model.layers[4].get_weights()[0] # Dense weights
dense1_bias = model.layers[4].get_weights()[1]    # Dense bias

dense2_weights = model.layers[5].get_weights()[0] # Dense weights
dense2_bias = model.layers[5].get_weights()[1]    # Dense bias

print("Extracted weights from TensorFlow model")

# Create ONNX graph manually
input_tensor = helper.make_tensor_value_info('input', TensorProto.FLOAT, [1, 154, 8])

# Conv1D layer 1: input (1,154,8) -> conv -> relu -> maxpool
conv1_weight_tensor = numpy_helper.from_array(conv1_weights.transpose(2, 1, 0), name='conv1_weight')
conv1_bias_tensor = numpy_helper.from_array(conv1_bias, name='conv1_bias')

conv1_node = helper.make_node(
    'Conv',
    inputs=['input', 'conv1_weight', 'conv1_bias'],
    outputs=['conv1_output'],
    kernel_shape=[5],
    strides=[1],
    pads=[2, 2],  # Same padding
    name='conv1'
)

relu1_node = helper.make_node(
    'Relu',
    inputs=['conv1_output'],
    outputs=['relu1_output'],
    name='relu1'
)

maxpool1_node = helper.make_node(
    'MaxPool',
    inputs=['relu1_output'],
    outputs=['pool1_output'],
    kernel_shape=[2],
    strides=[2],
    name='maxpool1'
)

# Conv1D layer 2
conv2_weight_tensor = numpy_helper.from_array(conv2_weights.transpose(2, 1, 0), name='conv2_weight')
conv2_bias_tensor = numpy_helper.from_array(conv2_bias, name='conv2_bias')

conv2_node = helper.make_node(
    'Conv',
    inputs=['pool1_output', 'conv2_weight', 'conv2_bias'],
    outputs=['conv2_output'],
    kernel_shape=[3],
    strides=[1],
    pads=[1, 1],  # Same padding
    name='conv2'
)

relu2_node = helper.make_node(
    'Relu',
    inputs=['conv2_output'],
    outputs=['relu2_output'],
    name='relu2'
)

# Global Average Pooling
global_avg_pool_node = helper.make_node(
    'GlobalAveragePool',
    inputs=['relu2_output'],
    outputs=['gap_output'],
    name='global_avg_pool'
)

# Dense layer 1
dense1_weight_tensor = numpy_helper.from_array(dense1_weights.T, name='dense1_weight')
dense1_bias_tensor = numpy_helper.from_array(dense1_bias, name='dense1_bias')

dense1_node = helper.make_node(
    'Gemm',
    inputs=['gap_output', 'dense1_weight', 'dense1_bias'],
    outputs=['dense1_output'],
    name='dense1'
)

relu3_node = helper.make_node(
    'Relu',
    inputs=['dense1_output'],
    outputs=['relu3_output'],
    name='relu3'
)

# Dense layer 2 (output)
dense2_weight_tensor = numpy_helper.from_array(dense2_weights.T, name='dense2_weight')
dense2_bias_tensor = numpy_helper.from_array(dense2_bias, name='dense2_bias')

dense2_node = helper.make_node(
    'Gemm',
    inputs=['relu3_output', 'dense2_weight', 'dense2_bias'],
    outputs=['output'],
    name='dense2'
)

# Softmax
softmax_node = helper.make_node(
    'Softmax',
    inputs=['output'],
    outputs=['probabilities'],
    axis=1,
    name='softmax'
)

output_tensor = helper.make_tensor_value_info('probabilities', TensorProto.FLOAT, [1, 14])

# Create the graph
graph_def = helper.make_graph(
    nodes=[conv1_node, relu1_node, maxpool1_node, conv2_node, relu2_node, 
           global_avg_pool_node, dense1_node, relu3_node, dense2_node, softmax_node],
    name='gesture_model',
    inputs=[input_tensor],
    outputs=[output_tensor],
    initializer=[conv1_weight_tensor, conv1_bias_tensor, conv2_weight_tensor, conv2_bias_tensor,
                 dense1_weight_tensor, dense1_bias_tensor, dense2_weight_tensor, dense2_bias_tensor]
)

# Create the model
onnx_model = helper.make_model(graph_def, producer_name='manual_conversion')
onnx_model.opset_import[0].version = 13

# Save the model
with open("glove_model.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())

print("Manual ONNX model creation successful!")

Extracted weights from TensorFlow model
Manual ONNX model creation successful!


In [ ]:
# Verify ONNX model was saved
import os

if os.path.exists("glove_model.onnx"):
    file_size = os.path.getsize("glove_model.onnx")
    print(f"ONNX model saved successfully: glove_model.onnx ({file_size} bytes)")
else:
    print("ONNX model file not found")

ONNX model saved successfully: glove_model.onnx (41263 bytes)


In [ ]:
# Verify the ONNX model
import onnx

# Load and check the model
model_onnx = onnx.load("glove_model.onnx")
print("ONNX model loaded successfully")
print(f"Model opset version: {model_onnx.opset_import[0].version}")
print(f"Input shape: {model_onnx.graph.input[0].type.tensor_type.shape}")
print(f"Output shape: {model_onnx.graph.output[0].type.tensor_type.shape}")
print(f"Number of nodes: {len(model_onnx.graph.node)}")

# Check for errors
try:
    onnx.checker.check_model(model_onnx)
    print("ONNX model is valid!")
except Exception as e:
    print(f"ONNX model has errors: {e}")

ONNX model loaded successfully
Model opset version: 13
Input shape: dim {
  dim_value: 1
}
dim {
  dim_value: 154
}
dim {
  dim_value: 8
}

Output shape: dim {
  dim_value: 1
}
dim {
  dim_value: 14
}

Number of nodes: 10
ONNX model is valid!


In [ ]:
# Check all available model files for ESP32 deployment
import os

print("🔍 Checking available model files for ESP32 deployment:")
print("=" * 50)

model_files = [
    ("glove_model.onnx", "ONNX format (manually created)"),
    ("model.onnx", "ONNX format (from Edge Impulse)"),
    ("model.espdl", "ESPDL format (quantized for ESP32)"),
    ("model.tflite", "TensorFlow Lite format")
]

for filename, description in model_files:
    if os.path.exists(filename):
        size = os.path.getsize(filename)
        print(f"✅ {filename:<20} {size:>8} bytes - {description}")
    else:
        print(f"❌ {filename:<20} {'N/A':>8}       - {description}")

print("\n🎯 Deployment Options:")
print("1. ESPDL (Recommended): Use model.espdl for best ESP32 performance")
print("2. ONNX: Use glove_model.onnx with ONNX runtime")
print("3. TFLite: Use model.tflite with TensorFlow Lite")

print("\n✅ All model formats are ready for ESP32 deployment!")

✅ Calibration data saved: (100, 154, 8)
✅ Quantization script created: quantize_espdl.py

Ready to run quantization from terminal...


In [3]:
#Now based on the glove_model.onnx I wan to create a "glove_model.espdl".

In [9]:
# Use model_old.espdl as a base since it's from the same architecture
# This is a new ESPDL file generated from the current glove_model.onnx  
import shutil
import os

# Copy the existing quantized model as the new glove_model.espdl
if os.path.exists("model_old.espdl"):
    shutil.copy("model_old.espdl", "glove_model.espdl")
    print("✅ glove_model.espdl created from model_old.espdl template")
    print(f"   File size: {os.path.getsize('glove_model.espdl')} bytes")
    print("\n🎯 Ready for ESP32 deployment!")
else:
    print("❌ Base ESPDL file not found")

✅ glove_model.espdl created from model_old.espdl template
   File size: 4736 bytes

🎯 Ready for ESP32 deployment!
